# Introduction

**[BERTopic](https://github.com/MaartenGr/BERTopic)** is a tool which visualizes the results of **topic modeling**.

BERTopic has many visualization methods.
- Visualize Topics(this one)
- Visualize Topic Hierarchy
- Visualize Topic Terms
- Visualize Topic Similarity
- Visualize Term Score Decline
- Visualize Topic Probability Distribution
- Visualize Topics over Time
- Visualize Topics per Clas

This notebook shows the visualization results.

I hope it would be helpful for data analysis somewhat.

*Note*
- *There is no submission file.*
- *BERTopic can do more than just visualization.For more detail, please check [BERTopic github](https://github.com/MaartenGr/BERTopic).*

# Data load

In [14]:
import re
from pathlib import Path

import jieba
import numpy as np
import pandas as pd
from nltk.corpus import stopwords

In [2]:
from pathlib import Path

workspace_root = Path('/Users/chengtechang/Desktop/vs')
if not workspace_root.exists():
    workspace_root = Path.cwd()

DATA_PATH = workspace_root / 'kaggle' / 'input' / 'commonlitreadabilityprize' / 'train.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'找不到資料檔案: {DATA_PATH}. 請將 notebook kernel 切換到 workspace 的 .venv。'
    )

train = pd.read_csv(DATA_PATH)

In [3]:
train

,id,url_legal,license,excerpt,target,standard_error
0,c12129c31,NaN,NaN,When the young people returned to the ballroom...,-0.340259,0.464009
1,85aa80a4c,NaN,NaN,"All through dinner time, Mrs. Fayre was somewh...",-0.315372,0.480805
2,b69ac6792,NaN,NaN,"As Roger had predicted, the snow departed as q...",-0.580118,0.476676
3,dd1000b26,NaN,NaN,And outside before the palace a great garden w...,-1.054013,0.450007
4,37c1b32fb,NaN,NaN,Once upon a time there were Three Bears who li...,0.247197,0.510845
...,...,...,...,...,...,...
2829,25ca8f498,https://sites.ehe.osu.edu/beyondpenguins/files...,CC BY-SA 3.0,When you think of dinosaurs and where they liv...,1.711390,0.646900
2830,2c26db523,https://en.wikibooks.org/wiki/Wikijunior:The_E...,CC BY-SA 3.0,So what is a solid? Solids are usually hard be...,0.189476,0.535648
2831,cd19e2350,https://en.wikibooks.org/wiki/Wikijunior:The_E...,CC BY-SA 3.0,The second state of matter we will discuss is ...,0.255209,0.483866
2832,15e2e9e7a,https://en.wikibooks.org/wiki/Geometry_for_Ele...,CC BY-SA 3.0,Solids are shapes that you can actually touch....,-0.215279,0.514128


In [15]:
ENGLISH_STOP_WORDS = set(stopwords.words("english"))
ENGLISH_STOP_WORDS.update({"also", "could", "would"})

CHINESE_STOP_WORDS = {
    "的", "了", "是", "在", "和", "與", "及", "或", "也", "都", "而", "就",
    "這", "這是", "那", "有", "沒有", "我", "你", "他", "她", "它", "我們", "你們",
}


def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = [token.strip() for token in jieba.cut(text, cut_all=False)]
    tokens = [
        token for token in tokens
        if token
        and token not in ENGLISH_STOP_WORDS
        and token not in CHINESE_STOP_WORDS
    ]
    return " ".join(tokens)


train["processed_excerpt"] = train["excerpt"].map(preprocess_text)
docs = train["processed_excerpt"].tolist()

print("原始:", train["excerpt"].iloc[0][:200])
print("前處理:", docs[0][:200])
print("文件數:", len(docs))

Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/sk/j94l_d853vx6kdmz54x33ytw0000gn/T/jieba.cache
Loading model cost 0.411 seconds.
Prefix dict has been built successfully.


原始: When the young people returned to the ballroom, it presented a decidedly changed appearance. Instead of an interior scene, it was a winter landscape.
The floor was covered with snow-white canvas, not 
前處理: young people returned ballroom presented decidedly changed appearance instead interior scene winter landscape floor covered snow white canvas laid smoothly rumpled bumps hillocks like real snow field 
文件數: 2834


*Comment*  
*If there are good ways to remove unnecessary words such as stopwords from each sentence, please tell me.*

# Bertopic

[BERTopic](https://github.com/MaartenGr/BERTopic) is a topic modeling technique.

In [16]:
from bertopic import BERTopic
from sklearn.datasets import fetch_20newsgroups
 
topic_model = BERTopic()
topics, _ = topic_model.fit_transform(docs)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7290.50it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,174,-1_water_light_energy_cells,"[water, light, energy, cells, plants, air, cal...",[renewable energy energy replenished easily ye...
1,0,2058,0_one_said_little_time,"[one, said, little, time, great, old, came, da...",[one day betty father said let us go town tomo...
2,1,130,1_brain_people_memory_different,"[brain, people, memory, different, cells, neur...",[scientists figured exactly people struggle fo...
3,2,111,2_water_iron_vessel_may,"[water, iron, vessel, may, gas, one, steam, oi...",[hydrochloric acid gas passes vessel suitable ...
4,3,84,3_computer_data_used_information,"[computer, data, used, information, software, ...",[operating system os name group computer progr...
5,4,83,4_species_food_animals_organisms,"[species, food, animals, organisms, plants, en...",[organism living thing easy recognise living t...
6,5,39,5_dna_bacteria_gene_antibiotics,"[dna, bacteria, gene, antibiotics, cells, dise...",[information microbe kind cell exists cell dna...
7,6,38,6_ice_earth_ocean_water,"[ice, earth, ocean, water, atmosphere, greenho...",[greenhouse effect occurs certain gases earth ...
8,7,37,7_earth_sun_stars_planets,"[earth, sun, stars, planets, solar, galaxy, ga...",[solar system sun objects orbit around sun orb...
9,8,36,8_current_electric_electricity_wire,"[current, electric, electricity, wire, magneti...",[copper even conductor melted powerful current...


In [8]:
topic_model.get_topic(0)

[('the', np.float64(0.04730773197854975)),
 ('and', np.float64(0.04023250647404587)),
 ('of', np.float64(0.034462736623546)),
 ('to', np.float64(0.03411170863309464)),
 ('was', np.float64(0.029208377009542423)),
 ('in', np.float64(0.027592122454719016)),
 ('he', np.float64(0.027248646005889516)),
 ('his', np.float64(0.02242542502722663)),
 ('it', np.float64(0.021812727973801255)),
 ('that', np.float64(0.020570151077256318))]

## Visualize Topics

In [9]:
topic_model.visualize_topics()

## Visualize Topic Hierarchy

In [10]:
topic_model.visualize_hierarchy()

## Visualize Topic Terms

In [11]:
topic_model.visualize_barchart()

## Visualize Topic Similarity

In [12]:
topic_model.visualize_heatmap()

## Visualize Term Score Decline	

In [13]:
topic_model.visualize_term_rank()